In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

# =========================
# CONFIG
# =========================
BASE_DIR = "/Users/kaitlinzareno/Desktop/csci535/csci535-project/data"

INPUT_DIR = os.path.join(BASE_DIR, "openface_new")
OUTPUT_BASE = os.path.join(BASE_DIR, "coordination", "openface")

TAUS = ['0100', '0200', '0400', '0500', '0800','1600']
TAU = TAUS[3]
LABELS_PATH = os.path.join(BASE_DIR, "labels_tau", f"labels_tau_{TAU}.json")
MANIFEST_PATH = os.path.join(BASE_DIR, "manifest.csv")

os.makedirs(OUTPUT_BASE, exist_ok=True)

WIN_SIZE = 60
STRIDE = 60
MAX_LAG = 10
FPS = 30
HIGH_SYNC_THRESHOLD = 0.5

TRAINING_CLASSES = {0, 1, 2}

In [ ]:
def load_labels(labels_path):
    """
    Load labels_tau_XXXX.json into {basename: class_int}.
    """
    VALID_LABEL_INTS = {0, 1, 2, 3, 4, 5}

    if not os.path.exists(labels_path):
        raise FileNotFoundError(f"labels JSON not found: {labels_path}")

    with open(labels_path, "r") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError(
            f"labels JSON must be a dict/object. Got {type(data).__name__}"
        )

    validated = {}

    for basename, raw in data.items():
        if not isinstance(basename, str):
            raise ValueError(f"Label key must be string. Got {basename!r}")

        try:
            cls = int(raw)
        except Exception as e:
            raise ValueError(
                f"Label value for {basename!r} is not int-convertible: {raw!r}"
            ) from e

        if cls not in VALID_LABEL_INTS:
            raise ValueError(
                f"Label value for {basename!r}={cls} outside valid range 0..5"
            )

        validated[basename] = cls

    return validated

In [ ]:
def basename_to_window(basename):
    import os, re

    b = os.path.basename(str(basename))
    b = b.replace(".npy", "").replace(".mp4", "").replace(".wav", "")

    pattern = (
        r"(?P<interaction>V\d+_S\d+_I\d+)_"
        r"(?P<participant>P\d+)_t_(?P<end>\d+)"
    )

    m = re.search(pattern, b)

    if m is None:
        raise ValueError(f"Could not parse: {basename}")

    interaction = m.group("interaction")
    participant = m.group("participant")
    end_raw = m.group("end")

    # convert ms → seconds
    end_s = int(end_raw) / 1000.0
    start_s = end_s - 2.0

    # round for safety
    start_s = round(start_s, 3)
    end_s = round(end_s, 3)

    # ✅ EXACT formatting
    start_str = f"{start_s:08.3f}"   # 0000.000
    end_str   = f"{end_s:08.3f}"     # 0002.000

    time_window = f"{start_str}-{end_str}"

    return participant, interaction, start_str, end_str

In [ ]:
def load_manifest(manifest_path):
    if not os.path.exists(manifest_path):
        raise FileNotFoundError(f"manifest CSV not found: {manifest_path}")

    df = pd.read_csv(manifest_path)

    print("Manifest columns:")
    print(list(df.columns))

    return df

def infer_col(df, candidates, required=True):
    """
    Find the first matching column from candidates.
    """
    cols = list(df.columns)
    lower_to_real = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_to_real:
            return lower_to_real[cand.lower()]

    if required:
        raise ValueError(
            f"Could not infer column from candidates {candidates}. "
            f"Available columns: {cols}"
        )

    return None

def build_fid_lookup(manifest_df):
    required = [
        "interaction_id",
        "split",
        "file_id_a",
        "file_id_b",
        "participant_a",
        "participant_b",
    ]

    missing = [c for c in required if c not in manifest_df.columns]
    if missing:
        raise ValueError(f"Manifest missing required columns: {missing}")

    lookup = {}

    for _, row in manifest_df.iterrows():
        interaction_id = str(row["interaction_id"])
        split = str(row["split"])

        file_id_a = str(row["file_id_a"]).replace(".mp4", "").replace(".npy", "").replace(".wav", "")
        file_id_b = str(row["file_id_b"]).replace(".mp4", "").replace(".npy", "").replace(".wav", "")

        participant_a = str(row["participant_a"])
        participant_b = str(row["participant_b"])

        # A is speaker -> B listener
        lookup[file_id_a] = {
            "interaction_id": interaction_id,
            "listener_file_id": file_id_b,
            "speaker_participant": participant_a,
            "listener_participant": participant_b,
            "split": split,
        }

        # B is speaker -> A listener
        lookup[file_id_b] = {
            "interaction_id": interaction_id,
            "listener_file_id": file_id_a,
            "speaker_participant": participant_b,
            "listener_participant": participant_a,
            "split": split,
        }

    return lookup

In [ ]:
def enumerate_samples(manifest_df, labels_dict, training_classes=TRAINING_CLASSES):
    fid_lookup = build_fid_lookup(manifest_df)
    training_classes = set(training_classes)

    rows = []

    missing_from_manifest = 0
    skipped_nontraining = 0
    parse_failed = 0

    for basename, raw_cls in labels_dict.items():
        cls = int(raw_cls)
        #breakpoint()

        if cls not in training_classes:
            skipped_nontraining += 1
            continue

        try:
            speaker_fid, interaction, start_s, end_s = basename_to_window(basename)
            #breakpoint()
        except Exception:
            parse_failed += 1
            continue
        #breakpoint()
        role_info = fid_lookup.get(f'{interaction}_{speaker_fid}')

        if role_info is None:
            missing_from_manifest += 1
            continue

        rows.append({
        "basename": basename,
        "speaker_file_id": speaker_fid,
        "listener_file_id": role_info["listener_file_id"],
        "interaction_id": role_info["interaction_id"],
        "split": role_info["split"],
        "start_s": float(start_s),
        "end_s": float(end_s),
        "label": cls,
        "speaker_participant": role_info["speaker_participant"],
        "listener_participant": role_info["listener_participant"],
    })

    samples_df = pd.DataFrame(rows)

    print("\nEnumeration summary:")
    print(f"usable samples:          {len(samples_df)}")
    print(f"skipped non-training:    {skipped_nontraining}")
    print(f"missing from manifest:   {missing_from_manifest}")
    print(f"parse failed:            {parse_failed}")

    return samples_df

In [ ]:
# =========================
# OPENFACE FILE INDEX
# =========================

import os, re
import pandas as pd
from glob import glob

def parse_openface_filename(fp):
    fname = os.path.basename(fp).replace(".npy", "")

    pattern = (
        r"(?P<start>\d+(?:\.\d+)?)-(?P<end>\d+(?:\.\d+)?)_"
        r"(?P<interaction>V\d+_S\d+_I\d+)_"
        r"(?P<participant>P\d+)"
    )

    m = re.search(pattern, fname)
    if m is None:
        raise ValueError(f"Could not parse OpenFace filename: {fp}")

    start_s = round(float(m.group("start")), 3)
    end_s = round(float(m.group("end")), 3)

    interaction_id = m.group("interaction")
    participant = m.group("participant")
    file_id = f"{interaction_id}_{participant}"

    return {
        "filepath": fp,
        "file_id": file_id,
        "interaction_id": interaction_id,
        "participant": participant,
        "start_s": start_s,
        "end_s": end_s,
        "start_window": f"{start_s:08.3f}",
        "end_window": f"{end_s:08.3f}",
        "time_window": f"{start_s:08.3f}-{end_s:08.3f}",
    }

def build_file_index(input_dir):
    files = glob(os.path.join(input_dir, "*", "*.npy"))
    rows = []

    for fp in files:
        try:
            rows.append(parse_openface_filename(fp))
        except Exception as e:
            print("Skipping:", fp)
            print(e)

    df = pd.DataFrame(rows)

    print("Found files:", len(files))
    print("Parsed rows:", len(df))
    print("Columns:", list(df.columns))

    return df

file_df = build_file_index(INPUT_DIR)
display(file_df.head())

In [ ]:
def safe_corr(a, b):
    if len(a) < 3 or len(b) < 3:
        return 0.0

    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0

    r = np.corrcoef(a, b)[0, 1]

    if np.isnan(r) or np.isinf(r):
        return 0.0

    return float(r)

def compute_wcc(x, y, win_size=60, stride=60, max_lag=15):
    T = min(len(x), len(y))
    x = x[:T]
    y = y[:T]

    if x.ndim != 2 or y.ndim != 2:
        raise ValueError(f"Expected 2D arrays. Got x={x.shape}, y={y.shape}")

    if x.shape[1] != y.shape[1]:
        raise ValueError(f"Feature dimension mismatch: x={x.shape}, y={y.shape}")

    if T < win_size:
        win_size = T
        stride = T

    D = x.shape[1]
    lags = np.arange(-max_lag, max_lag + 1)

    all_windows = []

    for start in range(0, T - win_size + 1, stride):
        xw = x[start:start + win_size]
        yw = y[start:start + win_size]

        corr = np.zeros((D, len(lags)), dtype=np.float32)

        for d in range(D):
            xf = xw[:, d]
            yf = yw[:, d]

            for li, lag in enumerate(lags):
                if lag < 0:
                    # speaker leads listener
                    corr[d, li] = safe_corr(xf[:lag], yf[-lag:])
                elif lag > 0:
                    # listener leads speaker --> listener at prior time point = listener leads 
                    corr[d, li] = safe_corr(xf[lag:], yf[:-lag])
                else:
                    # simultaneous synchrony
                    corr[d, li] = safe_corr(xf, yf)

        all_windows.append(corr)

    return np.stack(all_windows, axis=0), lags

def summarize_wcc(wcc, lags, high_sync_threshold=0.5, fps=30.0):
    """
    Returns speaker/listener-aware coordination features.
    """

    abs_wcc = np.abs(wcc)

    peak_idx = np.argmax(abs_wcc, axis=-1)  # (windows, features)

    max_abs_corr = np.take_along_axis(
        abs_wcc,
        peak_idx[..., None],
        axis=-1
    ).squeeze(-1)

    signed_peak_corr = np.take_along_axis(
        wcc,
        peak_idx[..., None],
        axis=-1
    ).squeeze(-1)

    best_lag = lags[peak_idx]

    zero_lag_idx = np.where(lags == 0)[0][0]
    zero_lag_corr = wcc[..., zero_lag_idx]

    features = {
        # strength
        "mean_max_abs_corr": np.mean(max_abs_corr),
        "std_max_abs_corr": np.std(max_abs_corr),
        "median_max_abs_corr": np.median(max_abs_corr),
        "max_abs_corr": np.max(max_abs_corr),

        # signed peak
        "mean_signed_peak_corr": np.mean(signed_peak_corr),
        "std_signed_peak_corr": np.std(signed_peak_corr),

        # zero-lag
        "mean_zero_lag_corr": np.mean(zero_lag_corr),
        "mean_abs_zero_lag_corr": np.mean(np.abs(zero_lag_corr)),
        "std_zero_lag_corr": np.std(zero_lag_corr),

        # speaker/listener lag
        "mean_best_lag_frames": np.mean(best_lag),
        "std_best_lag_frames": np.std(best_lag),
        "mean_abs_best_lag_frames": np.mean(np.abs(best_lag)),
        "mean_best_lag_sec": np.mean(best_lag) / fps,
        "mean_abs_best_lag_sec": np.mean(np.abs(best_lag)) / fps,

        # directionality
        "listener_lead_ratio": np.mean(best_lag > 0),
        "speaker_lead_ratio": np.mean(best_lag < 0),
        "zero_lag_ratio": np.mean(best_lag == 0),
        "listener_speaker_lead_score": (
            np.mean(best_lag > 0) - np.mean(best_lag < 0)
        ),

        # threshold
        "high_sync_ratio": np.mean(max_abs_corr > high_sync_threshold),
    }

    return features

In [ ]:
# =========================
# LOAD LABELS + MANIFEST
# =========================
labels_dict = load_labels(LABELS_PATH)
manifest_df = load_manifest(MANIFEST_PATH)

samples_df = enumerate_samples(
    manifest_df=manifest_df,
    labels_dict=labels_dict,
    training_classes=TRAINING_CLASSES,
)

print("\nSamples:")
display(samples_df.head())

# =========================
# LOAD OPENFACE INDEX
# =========================
file_df = build_file_index(INPUT_DIR)

print("\nOpenFace file index:")
display(file_df.head())

In [ ]:
# =========================
# MATCH SPEAKER/LISTENER TO OPENFACE FILES
# =========================

speaker_files = file_df.rename(columns={
    "filepath": "speaker_filepath",
    "participant": "speaker_participant",
})

listener_files = file_df.rename(columns={
    "filepath": "listener_filepath",
    "participant": "listener_participant",
})

speaker_files = speaker_files[
    [
        "interaction_id",
        "speaker_participant",
        "start_s",
        "end_s",
        "speaker_filepath",
    ]
]

listener_files = listener_files[
    [
        "interaction_id",
        "listener_participant",
        "start_s",
        "end_s",
        "listener_filepath",
    ]
]

matched = samples_df.merge(
    speaker_files,
    on=["interaction_id", "speaker_participant", "start_s", "end_s"],
    how="left",
)

matched = matched.merge(
    listener_files,
    on=["interaction_id", "listener_participant", "start_s", "end_s"],
    how="left",
)

print("\nMatched samples:")
display(matched.head())

print("\nBefore drop:", len(matched))
print("Missing speaker files:", matched["speaker_filepath"].isna().sum())
print("Missing listener files:", matched["listener_filepath"].isna().sum())

matched = matched.dropna(
    subset=["speaker_filepath", "listener_filepath"]
).copy()

print("Usable matched samples:", len(matched))

fast wcc


In [ ]:
def compute_wcc_fast(x, y, max_lag=15):
    T = min(len(x), len(y))
    x = x[:T]
    y = y[:T]

    # normalize once
    x = (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-8)
    y = (y - y.mean(axis=0)) / (y.std(axis=0) + 1e-8)

    lags = np.arange(-max_lag, max_lag + 1)
    D = x.shape[1]

    wcc = np.zeros((D, len(lags)), dtype=np.float32)

    for i, lag in enumerate(lags):
        if lag < 0:
            #speaker leads
            wcc[:, i] = np.mean(x[:lag] * y[-lag:], axis=0)
        elif lag > 0:
            #listener leads
            wcc[:, i] = np.mean(x[lag:] * y[:-lag], axis=0)
        else:
            wcc[:, i] = np.mean(x * y, axis=0)

    return wcc[None, ...], lags

def summarize_wcc(wcc, lags, high_sync_threshold=0.5, fps=30.0):
    abs_wcc = np.abs(wcc)

    peak_idx = np.argmax(abs_wcc, axis=-1)

    max_abs_corr = np.take_along_axis(
        abs_wcc, peak_idx[..., None], axis=-1
    ).squeeze(-1)

    signed_peak_corr = np.take_along_axis(
        wcc, peak_idx[..., None], axis=-1
    ).squeeze(-1)

    best_lag = lags[peak_idx]

    zero_lag_idx = np.where(lags == 0)[0][0]
    zero_lag_corr = wcc[..., zero_lag_idx]

    return {
        "mean_max_abs_corr": np.mean(max_abs_corr),
        "std_max_abs_corr": np.std(max_abs_corr),
        "median_max_abs_corr": np.median(max_abs_corr),
        "max_abs_corr": np.max(max_abs_corr),

        "mean_signed_peak_corr": np.mean(signed_peak_corr),
        "std_signed_peak_corr": np.std(signed_peak_corr),

        "mean_zero_lag_corr": np.mean(zero_lag_corr),
        "mean_abs_zero_lag_corr": np.mean(np.abs(zero_lag_corr)),
        "std_zero_lag_corr": np.std(zero_lag_corr),

        "mean_best_lag_frames": np.mean(best_lag),
        "std_best_lag_frames": np.std(best_lag),
        "mean_abs_best_lag_frames": np.mean(np.abs(best_lag)),
        "mean_best_lag_sec": np.mean(best_lag) / fps,
        "mean_abs_best_lag_sec": np.mean(np.abs(best_lag)) / fps,

        "listener_lead_ratio": np.mean(best_lag > 0),
        "speaker_lead_ratio": np.mean(best_lag < 0),
        "zero_lag_ratio": np.mean(best_lag == 0),
        "listener_speaker_lead_score": (
            np.mean(best_lag > 0) - np.mean(best_lag < 0)
        ),

        "high_sync_ratio": np.mean(max_abs_corr > high_sync_threshold),
    }

In [ ]:
import csv
import os

OUT_CSV = os.path.join(OUTPUT_BASE, f"all_coordination_features_tau_{TAU}.csv")

# write header once
first = True

with open(OUT_CSV, "w", newline="") as f:
    writer = None

    for i, (_, row) in enumerate(tqdm(matched.iterrows(), total=len(matched))):
        try:
            x = np.load(row["speaker_filepath"], mmap_mode='r')
            y = np.load(row["listener_filepath"], mmap_mode='r')

            wcc, lags = compute_wcc_fast(x, y, max_lag=MAX_LAG)

            coord_features = summarize_wcc(
                wcc, lags,
                high_sync_threshold=HIGH_SYNC_THRESHOLD,
                fps=FPS,
            )

            start_s = float(row["start_s"])
            end_s = float(row["end_s"])

            out_row = {
                "interaction_name": row["interaction_id"],
                "start_s": start_s,
                "end_s": end_s,
                "time_window": f"{start_s:08.3f}-{end_s:08.3f}",
                "speaker": row["speaker_participant"],
                "listener": row["listener_participant"],
                **coord_features,
            }

            # initialize writer once
            if writer is None:
                writer = csv.DictWriter(f, fieldnames=out_row.keys())
                writer.writeheader()

            writer.writerow(out_row)

            # flush periodically (important in Colab)
            if i % 500 == 0:
                f.flush()

        except Exception as e:
            continue

print("Done.")

Unsliced window coordination